In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/master_long_hourly_test_2014_05_12.csv" "/content/master_long_hourly_test_2014_05_12.csv"
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/master_long_hourly_train_2012_2013.csv" "/content/master_long_hourly_train_2012_2013.csv"
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/master_long_hourly_validation_2014_01_04.csv" "/content/master_long_hourly_validation_2014_01_04.csv"

In [3]:
!pip install statsforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [4]:
from pathlib import Path

# Input files
TRAIN_PATH = "/content/master_long_hourly_train_2012_2013.csv"
VAL_PATH   = "/content/master_long_hourly_validation_2014_01_04.csv"
TEST_PATH  = "/content/master_long_hourly_test_2014_05_12.csv"

# Output
SELECTED_CLIENTS_PATH = Path("selected_clients_autoets.csv")

# Sampling
RANDOM_SEED = 42

# Model
SEASON_LENGTH  = 24   # hourly data → daily seasonality
LOOKBACK_HOURS = 672  # 4 weeks lookback
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [5]:
# IMPORTS
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

In [6]:
# ── SAMPLE CLIENTS ────────────────────────────────────────────────────────────
train_ids = pd.read_csv(TRAIN_PATH, usecols=["client_id"])
all_clients = train_ids["client_id"].unique()

selected = sorted(all_clients.tolist())  # Use all clients

print(f"Selected {len(selected)} clients (ALL)")

pd.DataFrame({"client_id": selected}).to_csv(SELECTED_CLIENTS_PATH, index=False)
print(f"Saved client list → {SELECTED_CLIENTS_PATH}")

Selected 156 clients (ALL)
Saved client list → selected_clients_autoets.csv


In [7]:
# ── PREPARE DATA ──────────────────────────────────────────────────────────────
# Pure endogenous: only unique_id, ds, y — no exogenous features

def load_split(path, clients):
    df = pd.read_csv(path, parse_dates=["timestamp"])
    df = df[df["client_id"].isin(clients)].copy()
    # StatsForecast expects: unique_id, ds, y
    df = df.rename(columns={"client_id": "unique_id", "timestamp": "ds"})
    return df[["unique_id", "ds", "y"]].sort_values(["unique_id", "ds"]).reset_index(drop=True)


def apply_lookback(df, hours):
    """Keep only the last `hours` rows per client."""
    return (df.groupby("unique_id", group_keys=False)
              .tail(hours)
              .reset_index(drop=True))


train = load_split(TRAIN_PATH, selected)
val   = load_split(VAL_PATH,   selected)
test  = load_split(TEST_PATH,  selected)

# Apply 4-week lookback
train = apply_lookback(train, LOOKBACK_HOURS)

print(f"train : {train.shape}  {train['ds'].min()} → {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} → {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} → {test['ds'].max()}")

train : (104832, 3)  2013-12-04 00:00:00 → 2013-12-31 23:00:00
val   : (445536, 3)    2014-01-01 00:00:00 → 2014-04-30 23:00:00
test  : (913536, 3)   2014-05-01 00:00:00 → 2014-12-31 23:00:00


In [8]:
# ── EVALUATION METRICS ────────────────────────────────────────────────────────
def compute_metrics(df, target_col="y", pred_col="AutoETS"):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {"MSE": round(mse, 4), "MAE": round(mae, 4), "WAPE": round(wape, 4)}


def per_client_metrics(df, target_col="y", pred_col="AutoETS"):
    rows = []
    for uid, grp in df.groupby("unique_id"):
        rows.append({"client_id": uid, **compute_metrics(grp, target_col, pred_col)})
    rows.append({"client_id": "OVERALL", **compute_metrics(df, target_col, pred_col)})
    return pd.DataFrame(rows)

In [9]:
# ── TRAIN AutoETS (Pure Endogenous) ───────────────────────────────────────────
# AutoETS automatically selects the best error, trend, seasonality components
model = AutoETS(
    season_length=SEASON_LENGTH,  # daily seasonality
)

sf = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf.fit(train[["unique_id", "ds", "y"]])

joblib.dump(sf, MODEL_DIR / "autoets_val.joblib")
print("Training complete. Model saved → models/autoets_val.joblib")

Training complete. Model saved → models/autoets_val.joblib


In [10]:
# ── VALIDATION FORECAST ───────────────────────────────────────────────────────
val_h = val["ds"].nunique()

val_preds = sf.predict(h=val_h)
val_eval  = val[["unique_id", "ds", "y"]].merge(val_preds, on=["unique_id", "ds"])

print(val_eval.head())

  unique_id                  ds          y     AutoETS
0    MT_124 2014-01-01 00:00:00  95.693780  228.928874
1    MT_124 2014-01-01 01:00:00  83.732056  192.990966
2    MT_124 2014-01-01 02:00:00  82.535890  175.046857
3    MT_124 2014-01-01 03:00:00  82.535890  140.319270
4    MT_124 2014-01-01 04:00:00  63.397133   69.674424


In [11]:
# 1. Training Metrics (In-Sample)
train_preds_list = []
for i, uid in enumerate(train["unique_id"].unique()):
    fitted_model = sf.fitted_[i, 0]
    client_train = train[train["unique_id"] == uid].copy()

    in_sample_preds = fitted_model.predict_in_sample()

    if isinstance(in_sample_preds, dict):
        preds = in_sample_preds.get("mean", in_sample_preds.get("fitted", list(in_sample_preds.values())[0]))
    else:
        preds = in_sample_preds

    pred_df = pd.DataFrame({
        "unique_id": [uid] * len(client_train),
        "ds": client_train["ds"].values,
        "AutoETS": preds
    })
    train_preds_list.append(pred_df)

train_preds = pd.concat(train_preds_list, ignore_index=True)
train_eval  = train[["unique_id", "ds", "y"]].merge(train_preds, on=["unique_id", "ds"], how="left")

train_metrics = per_client_metrics(train_eval.dropna(), pred_col="AutoETS")
print("── Training Metrics (In-Sample) ──")
print(train_metrics.head(5).to_string(index=False))
print("...")
print(train_metrics.tail(1).to_string(index=False))
print("\n")

# 2. Validation Metrics (Out-of-Sample)
val_metrics = per_client_metrics(val_eval, pred_col="AutoETS")
print("── Validation Metrics (Out-of-Sample) ──")
print(val_metrics.head(5).to_string(index=False))
print("...")
print(val_metrics.tail(1).to_string(index=False))

── Training Metrics (In-Sample) ──
client_id       MSE     MAE   WAPE
   MT_124 2542.4793 36.1130 0.1349
   MT_132   98.1212  3.6690 0.2258
   MT_156  112.9570  7.6395 0.1138
   MT_158  565.4965 14.7422 0.1982
   MT_159  275.1525  8.7603 0.1427
...
client_id        MSE     MAE   WAPE
  OVERALL 44498.7452 45.7989 0.0736


── Validation Metrics (Out-of-Sample) ──
client_id        MSE     MAE   WAPE
   MT_124 11689.9045 97.1508 0.3588
   MT_132   548.0159 19.1826 1.1218
   MT_156  3080.7761 46.1875 0.5657
   MT_158  4106.5718 49.0217 0.6644
   MT_159  2452.9660 35.9695 0.7065
...
client_id          MSE      MAE   WAPE
  OVERALL 2550950.2635 317.2161 0.5172


In [12]:
# ── TEST EVALUATION ───────────────────────────────────────────────────────────
# Retrain on train+val (with lookback), then predict on held-out test set
train_val = pd.concat([train, val], ignore_index=True).sort_values(["unique_id", "ds"])
train_val = apply_lookback(train_val, LOOKBACK_HOURS)

model = AutoETS(
    season_length=SEASON_LENGTH,
)

sf_final = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf_final.fit(train_val[["unique_id", "ds", "y"]])

joblib.dump(sf_final, MODEL_DIR / "autoets_final.joblib")
print("Final model saved → models/autoets_final.joblib")

test_h     = test["ds"].nunique()
test_preds = sf_final.predict(h=test_h)
test_eval  = test[["unique_id", "ds", "y"]].merge(test_preds, on=["unique_id", "ds"])

test_metrics = per_client_metrics(test_eval, pred_col="AutoETS")
print("── Test metrics ──")
print(test_metrics.to_string(index=False))

Final model saved → models/autoets_final.joblib
── Test metrics ──
client_id          MSE       MAE    WAPE
   MT_124 4.696071e+03   52.0796  0.1734
   MT_132 4.570508e+02   15.1342  1.0000
   MT_156 9.054761e+02   25.3854  0.2854
   MT_158 3.522640e+03   50.8642  0.6490
   MT_159 1.281741e+03   32.2770  0.7538
   MT_161 7.572671e+04  212.4620  0.1355
   MT_162 4.601999e+07 5924.5957 21.8544
   MT_163 1.970569e+05  349.7080  0.1399
   MT_166 1.673822e+04   96.9028  0.0750
   MT_168 1.544406e+02    9.4222  0.0704
   MT_169 2.890180e+01    4.3964  0.1043
   MT_171 4.879428e+02   16.9951  0.0720
   MT_172 3.769472e+02   14.7641  0.1208
   MT_174 7.407025e+02   18.3544  0.1117
   MT_175 4.736323e+02   15.1661  0.0954
   MT_176 1.968331e+02   10.4060  0.1206
   MT_180 8.324832e+02   22.7562  0.1523
   MT_182 4.732400e+01    4.9865  0.0990
   MT_183 1.436664e+02    8.8508  0.1143
   MT_187 3.209346e+02   13.6990  0.1474
   MT_188 2.933430e+01    3.6157  0.0853
   MT_189 4.196598e+03   47.185